In [1]:
import tensorflow as tf
from tensorflow.keras import models, layers
import matplotlib.pyplot as plt
from sklearn import metrics

In [2]:
IMAGE_H= 224
IMAGE_W=224
BATCH_SIZE = 32
channels = 3
epochs= 50

In [3]:
dataset_directory = r"E:\ORAL_CANCER\1ST_DATASET"
dataset = tf.keras.preprocessing.image_dataset_from_directory(
    dataset_directory,
    shuffle=True,
    image_size=(IMAGE_H, IMAGE_W),
    batch_size=BATCH_SIZE
)

Found 8143 files belonging to 2 classes.


In [4]:
import tensorflow as tf
import os

# Constants
IMAGE_H = 224
IMAGE_W = 224
BATCH_SIZE = 32
SEED = 123

# Get the number of total images in the dataset
image_count = sum([len(files) for r, d, files in os.walk(dataset_directory) if files])
print(f"Total images found: {image_count}")

# Compute how many images go into each split
train_image_count = int(0.8 * image_count)
val_image_count = int(0.1 * image_count)
test_image_count = image_count - train_image_count - val_image_count

print(f"Training images: {train_image_count}")
print(f"Validation images: {val_image_count}")
print(f"Test images: {test_image_count}")

# Load the full dataset
full_dataset = tf.keras.preprocessing.image_dataset_from_directory(
    dataset_directory,
    shuffle=True,
    image_size=(IMAGE_H, IMAGE_W),
    batch_size=BATCH_SIZE,
    seed=SEED
)

# Split the dataset by batches
total_batches = len(full_dataset)
train_batches = int(0.8 * total_batches)
val_batches = int(0.1 * total_batches)
test_batches = total_batches - train_batches - val_batches

train_dataset = full_dataset.take(train_batches)
val_dataset = full_dataset.skip(train_batches).take(val_batches)
test_dataset = full_dataset.skip(train_batches + val_batches)

# Prefetch for performance
AUTOTUNE = tf.data.AUTOTUNE
train_dataset = train_dataset.prefetch(buffer_size=AUTOTUNE)
val_dataset = val_dataset.prefetch(buffer_size=AUTOTUNE)
test_dataset = test_dataset.prefetch(buffer_size=AUTOTUNE)

print(f"\nBatches:")
print(f"Training data: {train_batches} batches")
print(f"Validation data: {val_batches} batches")
print(f"Test data: {test_batches} batches")


Total images found: 8143
Training images: 6514
Validation images: 814
Test images: 815
Found 8143 files belonging to 2 classes.

Batches:
Training data: 204 batches
Validation data: 25 batches
Test data: 26 batches


In [5]:
AUTOTUNE = tf.data.AUTOTUNE
train_dataset = train_dataset.cache().shuffle(1000).prefetch(buffer_size=AUTOTUNE)

In [6]:
BATCH_SIZE = 32
IMAGE_H, IMAGE_W = 224, 224
CHANNELS = 3
N_CLASSES = 2

model = models.Sequential([
    layers.Conv2D(32, kernel_size=(5, 5), activation='relu', input_shape=(IMAGE_H, IMAGE_W, CHANNELS)),
    layers.Conv2D(32, kernel_size=(5, 5), activation='relu'),
    layers.MaxPooling2D((2, 2)),
    layers.Dropout(0.40),
    layers.Conv2D(64, kernel_size=(5, 5), activation='relu'),
    layers.Conv2D(64, kernel_size=(5, 5), activation='relu'),
    layers.Conv2D(64, kernel_size=(5, 5), activation='relu'),
    layers.MaxPooling2D((2, 2)),
    layers.Dropout(0.40),
    layers.Conv2D(128, kernel_size=(5, 5), activation='relu'),
    layers.Conv2D(128, kernel_size=(5, 5), activation='relu'),
    layers.Conv2D(128, kernel_size=(5, 5), activation='relu'),
    layers.MaxPooling2D((2, 2)),
    layers.Flatten(),
    layers.Dense(256, activation='relu'),
    layers.Dense(128, activation='relu'),
    layers.Dense(64, activation='relu'),
    layers.Dense(32, activation='relu'),
    layers.Dense(N_CLASSES, activation='softmax'),
])

In [7]:
model.summary()

Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 conv2d (Conv2D)             (None, 220, 220, 32)      2432      
                                                                 
 conv2d_1 (Conv2D)           (None, 216, 216, 32)      25632     
                                                                 
 max_pooling2d (MaxPooling2D  (None, 108, 108, 32)     0         
 )                                                               
                                                                 
 dropout (Dropout)           (None, 108, 108, 32)      0         
                                                                 
 conv2d_2 (Conv2D)           (None, 104, 104, 64)      51264     
                                                                 
 conv2d_3 (Conv2D)           (None, 100, 100, 64)      102464    
                                                        

In [8]:
loss_fn = tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True)

In [9]:
optimizer = tf.keras.optimizers.Adam(learning_rate=0.0001)

In [10]:
model.compile(optimizer=optimizer, loss=loss_fn, metrics=['accuracy'])

In [12]:
history=model.fit(
    train_dataset,
    epochs=epochs,
    batch_size=BATCH_SIZE,
    verbose=1,
    validation_data=val_dataset,
)

Epoch 1/50


C:\Users\gaura\anaconda3\envs\my_env\lib\site-packages\keras\backend.py:5582: UserWarning: "`sparse_categorical_crossentropy` received `from_logits=True`, but the `output` argument was produced by a Softmax activation and thus does not represent logits. Was this intended?
  output, from_logits = _get_logits(


204/204 [==============================] - 76s 280ms/step - loss: 0.8130 - accuracy: 0.5182 - val_loss: 0.6930 - val_accuracy: 0.5213
Epoch 2/50
204/204 [==============================] - 57s 277ms/step - loss: 0.6915 - accuracy: 0.5337 - val_loss: 0.6906 - val_accuracy: 0.5550
Epoch 3/50
204/204 [==============================] - 57s 277ms/step - loss: 0.6906 - accuracy: 0.5388 - val_loss: 0.6929 - val_accuracy: 0.5088
Epoch 4/50
204/204 [==============================] - 57s 278ms/step - loss: 0.6900 - accuracy: 0.5386 - val_loss: 0.6909 - val_accuracy: 0.5263
Epoch 5/50
204/204 [==============================] - 57s 280ms/step - loss: 0.6890 - accuracy: 0.5380 - val_loss: 0.6923 - val_accuracy: 0.5188
Epoch 6/50
204/204 [==============================] - 57s 278ms/step - loss: 0.6904 - accuracy: 0.5403 - val_loss: 0.6917 - val_accuracy: 0.5375
Epoch 7/50
204/204 [==============================] - 57s 278ms/step - loss: 0.6882 - accuracy: 0.5404 - val_loss: 0.6873 - val_accuracy: 0.5

In [13]:
model.evaluate(test_dataset)

26/26 [==============================] - 10s 161ms/step - loss: 0.9931 - accuracy: 0.7853


[0.9931347966194153, 0.7852760553359985]

In [14]:
acc= history.history['accuracy']
val_acc = history.history['val_accuracy']

loss= history.history['loss']
val_loss = history.history['val_loss']

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Set the style
sns.set(style="whitegrid")

# Assuming you have already imported necessary libraries and have data for epochs, acc, val_acc, loss, and val_loss

plt.figure(figsize=(16, 8))  # Increase figure size for better visualization

# Subplot 1: Training and Validation Accuracy
plt.subplot(1, 2, 1)
sns.lineplot(x=range(epochs), y=acc, label='Training Accuracy', linewidth=2, color='blue')
sns.lineplot(x=range(epochs), y=val_acc, label='Validation Accuracy', linewidth=2, color='orange')
plt.xlabel('Epochs')
plt.ylabel('Accuracy')
plt.title('Training and Validation Accuracy')
plt.legend(loc='lower right')
plt.grid(True)

# Subplot 2: Training and Validation Loss
plt.subplot(1, 2, 2)
sns.lineplot(x=range(epochs), y=loss, label='Training Loss', linewidth=2, color='blue')
sns.lineplot(x=range(epochs), y=val_loss, label='Validation Loss', linewidth=2, color='orange')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.title('Training and Validation Loss')
plt.legend(loc='upper right')
plt.grid(True)

plt.tight_layout()  # Adjust layout to prevent overlap
plt.show()
